In [78]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from tqdm.autonotebook import tqdm

## 原始数据提取

In [42]:
root_path = "./raw_data/"
data_path = os.listdir(root_path)

# 先进行LZ_WEST地区的电价预测
SettlementPointName = 'LZ_WEST'
SettlementPointType = 'LZ'

In [43]:
df_all = []
# 不清楚什么原因读取的特别慢（数据源都很小）
for i, v in enumerate(tqdm(sorted(data_path))):
    if v[0] == '.':
        continue
    for j in range(12):
        df = pd.read_excel(os.path.join(root_path, v), sheet_name=j)
        df = df[(df['Settlement Point Name'] == SettlementPoint) & (df['Settlement Point Type'] == SettlementPointType)]
        df = df[['Delivery Date', 'Delivery Hour', 'Delivery Interval', 'Settlement Point Price']]
        df_all.append(df)

  0%|          | 0/15 [00:00<?, ?it/s]

In [45]:
df = pd.concat(df_all)

In [58]:
df.to_csv('raw_data/processed_data.csv', index=None)

## 特征提取

In [75]:
"""
ERCOT RTM LMP 1小时预测特征工程模块
=====================================
预测目标: 未来15/30/45/60分钟的RTM电价
数据频率: 15分钟
滚动频率: 每15分钟预测一次

作者: 李源
日期: 2025-12-15
"""

import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# 美国联邦假日 (可通过 holidays 库获取更完整列表)
try:
    import holidays
    US_HOLIDAYS = holidays.US(years=range(2010, 2026))
except ImportError:
    US_HOLIDAYS = None


class ERCOTFeatureEngineer:
    """ERCOT RTM LMP 特征工程类"""
    
    # 预测horizon配置 (15分钟频率)
    HORIZONS = {
        'h1': 1,   # 15分钟后
        'h2': 2,   # 30分钟后
        'h3': 3,   # 45分钟后
        'h4': 4,   # 60分钟后
    }
    
    # ERCOT峰时定义
    PEAK_HOURS = list(range(7, 23))  # HE7-HE22
    SUMMER_MONTHS = [6, 7, 8, 9]
    SUPER_PEAK_HOURS = list(range(14, 21))  # HE14-HE20
    
    # 尖峰阈值 ($/MWh)
    SPIKE_THRESHOLDS = [100, 200, 500, 1000]
    
    def __init__(self, df: pd.DataFrame):
        """
        初始化特征工程器
        
        Parameters
        ----------
        df : pd.DataFrame
            原始数据，包含4列: date, hour, interval, price
        """
        self.df = self._preprocess(df.copy())
        
    def _preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        """数据预处理：构建时间戳索引"""
        df.columns = ['date', 'hour', 'interval', 'price']
        
        # 构建完整时间戳
        df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')
        # interval 1-4 对应 0, 15, 30, 45 分钟
        df['minute'] = (df['interval'] - 1) * 15
        # hour 1-24 转换为 0-23
        df['hour_adj'] = df['hour'] - 1
        
        df['timestamp'] = pd.to_datetime(
            df['date'].dt.strftime('%Y-%m-%d') + ' ' + 
            df['hour_adj'].astype(str).str.zfill(2) + ':' +
            df['minute'].astype(str).str.zfill(2)
        )
        
        df = df.set_index('timestamp').sort_index()
        df = df[['price']].copy()
        
        return df
    
    # =========================================================================
    # 时间特征
    # =========================================================================
    
    def _add_time_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加时间特征"""
        idx = df.index
        
        # 基础时间特征
        df['hour'] = idx.hour
        df['minute'] = idx.minute
        df['day_of_week'] = idx.dayofweek
        df['day_of_month'] = idx.day
        df['month'] = idx.month
        df['quarter'] = idx.quarter
        df['week_of_year'] = idx.isocalendar().week.astype(int)
        
        # 二值特征
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['is_peak_hour'] = df['hour'].isin(self.PEAK_HOURS).astype(int)
        df['is_summer'] = df['month'].isin(self.SUMMER_MONTHS).astype(int)
        df['is_super_peak'] = (
            df['is_summer'] & 
            df['hour'].isin(self.SUPER_PEAK_HOURS) &
            ~df['is_weekend'].astype(bool)
        ).astype(int)
        
        # 假日特征
        if US_HOLIDAYS is not None:
            holiday_dates = set(US_HOLIDAYS.keys())
            df['is_holiday'] = pd.Series(idx.date, index=idx).apply(
                lambda x: 1 if x in holiday_dates else 0
            )
        else:
            # 简化版: 标记主要假日
            df['is_holiday'] = self._simple_holiday_flag(idx)
        
        # 工作日特征
        df['is_business_day'] = (
            ~df['is_weekend'].astype(bool) & 
            ~df['is_holiday'].astype(bool)
        ).astype(int)
        
        # 周期编码 (避免边界跳跃)
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
        df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)
        df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
        df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
        
        # 时段特征
        df['hour_block'] = pd.cut(
            df['hour'], 
            bins=[-1, 6, 12, 18, 24], 
            labels=['night', 'morning', 'afternoon', 'evening']
        ).astype(str)
        
        return df
    
    def _simple_holiday_flag(self, idx: pd.DatetimeIndex) -> pd.Series:
        """简化版假日标记"""
        holidays_list = []
        for year in idx.year.unique():
            # 主要美国假日
            holidays_list.extend([
                f'{year}-01-01',  # 新年
                f'{year}-07-04',  # 独立日
                f'{year}-12-25',  # 圣诞节
                f'{year}-12-31',  # 新年前夜
            ])
        holidays_set = set(pd.to_datetime(holidays_list).date)
        return pd.Series(idx.date, index=idx).isin(holidays_set).astype(int)
    
    # =========================================================================
    # 滞后特征
    # =========================================================================
    
    def _add_lag_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """
        添加滞后特征
        
        Parameters
        ----------
        horizon : int
            预测步数，决定最小滞后量以避免数据泄露
        """
        price = df['price']
        min_lag = horizon  # 最小滞后量 = 预测步数
        
        # 短期滞后 (确保 >= min_lag)
        short_lags = [1, 2, 3, 4]  # 15, 30, 45, 60分钟
        for lag in short_lags:
            actual_lag = max(lag, min_lag)
            df[f'price_lag_{lag}'] = price.shift(actual_lag)
        
        # 日周期滞后 (同时刻)
        df['price_lag_96'] = price.shift(96)    # 24小时前 (96个15分钟)
        df['price_lag_192'] = price.shift(192)  # 48小时前
        df['price_lag_288'] = price.shift(288)  # 72小时前
        
        # 周周期滞后
        df['price_lag_672'] = price.shift(672)   # 7天前 (96*7)
        df['price_lag_1344'] = price.shift(1344) # 14天前
        
        # 年周期滞后 (近似)
        df['price_lag_35040'] = price.shift(35040)  # 约365天 (96*365)
        
        return df
    
    # =========================================================================
    # 滚动统计特征
    # =========================================================================
    
    def _add_rolling_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加滚动统计特征"""
        price = df['price']
        min_shift = horizon
        
        # 定义窗口大小 (15分钟频率)
        windows = {
            '1h': 4,      # 1小时 = 4个15分钟
            '3h': 12,     # 3小时
            '6h': 24,     # 6小时
            '24h': 96,    # 24小时
            '7d': 672,    # 7天
            '14d': 1344,  # 14天
        }
        
        for name, window in windows.items():
            shifted = price.shift(min_shift)
            rolling = shifted.rolling(window=window, min_periods=max(1, window // 2))
            
            df[f'price_mean_{name}'] = rolling.mean()
            df[f'price_std_{name}'] = rolling.std()
            df[f'price_max_{name}'] = rolling.max()
            df[f'price_min_{name}'] = rolling.min()
            
            # 仅对较大窗口计算高阶统计量
            if window >= 24:
                df[f'price_median_{name}'] = rolling.median()
                df[f'price_skew_{name}'] = rolling.skew()
                df[f'price_range_{name}'] = df[f'price_max_{name}'] - df[f'price_min_{name}']
        
        return df
    
    # =========================================================================
    # 同时刻历史统计特征
    # =========================================================================
    
    def _add_same_time_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加同时刻历史统计特征"""
        df_temp = df.copy()
        df_temp['hour'] = df_temp.index.hour
        df_temp['minute'] = df_temp.index.minute
        df_temp['dow'] = df_temp.index.dayofweek
        
        # 同时刻同星期 (过去4周)
        df['same_time_dow_mean_4w'] = df_temp.groupby(['hour', 'minute', 'dow'])['price'].transform(
            lambda x: x.shift(horizon).rolling(window=4, min_periods=1).mean()
        )
        df['same_time_dow_std_4w'] = df_temp.groupby(['hour', 'minute', 'dow'])['price'].transform(
            lambda x: x.shift(horizon).rolling(window=4, min_periods=1).std()
        )
        
        # 同时刻 (不区分星期, 过去7天)
        df['same_time_mean_7d'] = df_temp.groupby(['hour', 'minute'])['price'].transform(
            lambda x: x.shift(horizon).rolling(window=7, min_periods=1).mean()
        )
        
        return df
    
    # =========================================================================
    # 价格变化/动量特征
    # =========================================================================
    
    def _add_momentum_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加价格变化和动量特征"""
        price = df['price']
        min_shift = horizon
        
        shifted = price.shift(min_shift)
        
        # 差分
        df['price_diff_1'] = shifted.diff(1)      # 与前1个间隔的差
        df['price_diff_4'] = shifted.diff(4)      # 与1小时前的差
        df['price_diff_96'] = shifted.diff(96)    # 与24小时前的差
        
        # 变化率
        df['price_pct_1'] = shifted.pct_change(1)
        df['price_pct_4'] = shifted.pct_change(4)
        df['price_pct_96'] = shifted.pct_change(96)
        
        # 动量指标
        mean_1h = shifted.rolling(4).mean()
        std_1h = shifted.rolling(4).std()
        df['price_momentum_1h'] = (shifted - mean_1h) / (std_1h + 1e-6)
        
        mean_24h = shifted.rolling(96).mean()
        std_24h = shifted.rolling(96).std()
        df['price_momentum_24h'] = (shifted - mean_24h) / (std_24h + 1e-6)
        
        # 加速度 (二阶差分)
        df['price_acceleration'] = df['price_diff_1'].diff(1)
        
        # 趋势指标
        df['price_vs_24h_mean'] = shifted / (mean_24h + 1e-6)
        df['price_vs_7d_mean'] = shifted / (shifted.rolling(672).mean() + 1e-6)
        
        return df
    
    # =========================================================================
    # 尖峰/极值特征
    # =========================================================================
    
    def _add_spike_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加尖峰和极值特征"""
        price = df['price']
        min_shift = horizon
        shifted = price.shift(min_shift)
        
        # 尖峰标记 (多阈值)
        for threshold in self.SPIKE_THRESHOLDS:
            is_spike = (shifted > threshold).astype(int)
            df[f'is_spike_{threshold}'] = is_spike
            
            # 过去N小时尖峰次数
            df[f'spike_count_1h_{threshold}'] = is_spike.rolling(4).sum()
            df[f'spike_count_24h_{threshold}'] = is_spike.rolling(96).sum()
            df[f'spike_count_7d_{threshold}'] = is_spike.rolling(672).sum()
        
        # 距离上次尖峰的间隔数 (使用$100阈值)
        is_spike_100 = (shifted > 100).astype(bool)
        spike_positions = np.where(is_spike_100)[0]
        intervals_since = np.full(len(df), np.nan)
        
        last_spike = -np.inf
        for i in range(len(df)):
            if i in spike_positions:
                last_spike = i
            if last_spike >= 0:
                intervals_since[i] = i - last_spike
        df['intervals_since_spike'] = intervals_since
        
        # 极值特征
        df['price_percentile_24h'] = shifted.rolling(96).apply(
            lambda x: pd.Series(x).rank(pct=True).iloc[-1] if len(x) > 0 else np.nan,
            raw=False
        )
        
        rolling_max_24h = shifted.rolling(96).max()
        rolling_min_24h = shifted.rolling(96).min()
        df['price_vs_24h_max'] = shifted / (rolling_max_24h + 1e-6)
        df['price_vs_24h_min'] = shifted / (rolling_min_24h + 1e-6)
        
        rolling_max_7d = shifted.rolling(672).max()
        df['price_vs_7d_max'] = shifted / (rolling_max_7d + 1e-6)
        
        return df
    
    # =========================================================================
    # 波动率特征
    # =========================================================================
    
    def _add_volatility_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加波动率特征"""
        price = df['price']
        min_shift = horizon
        shifted = price.shift(min_shift)
        
        # 历史波动率 (标准差)
        df['volatility_1h'] = shifted.rolling(4).std()
        df['volatility_6h'] = shifted.rolling(24).std()
        df['volatility_24h'] = shifted.rolling(96).std()
        
        # 相对波动率
        mean_24h = shifted.rolling(96).mean()
        df['cv_24h'] = df['volatility_24h'] / (mean_24h + 1e-6)  # 变异系数
        
        # 价格振幅
        df['amplitude_1h'] = shifted.rolling(4).max() - shifted.rolling(4).min()
        df['amplitude_24h'] = shifted.rolling(96).max() - shifted.rolling(96).min()
        
        # 波动率变化
        df['volatility_change'] = df['volatility_1h'] - df['volatility_1h'].shift(4)
        
        return df
    
    # =========================================================================
    # Horizon特征 (用于单模型多输出方案)
    # =========================================================================
    
    def _add_horizon_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加预测horizon特征"""
        df['horizon'] = horizon
        df['horizon_minutes'] = horizon * 15
        
        # 目标时刻的时间特征
        target_hour = (df.index.hour + (df.index.minute + horizon * 15) // 60) % 24
        df['target_hour'] = target_hour
        df['target_minute'] = (df.index.minute + horizon * 15) % 60
        df['target_is_peak'] = target_hour.isin(self.PEAK_HOURS).astype(int)
        df['crosses_hour'] = ((df.index.minute + horizon * 15) >= 60).astype(int)
        
        return df
    
    # =========================================================================
    # 主入口：生成特征
    # =========================================================================
    
    def build_features(self, horizon: int) -> pd.DataFrame:
        """
        为指定horizon构建完整特征集
        
        Parameters
        ----------
        horizon : int
            预测步数 (1=15分钟, 2=30分钟, 3=45分钟, 4=60分钟)
        
        Returns
        -------
        pd.DataFrame
            包含所有特征的DataFrame
        """
        df = self.df.copy()
        
        # 依次添加各类特征
        df = self._add_time_features(df)
        df = self._add_lag_features(df, horizon)
        df = self._add_rolling_features(df, horizon)
        df = self._add_same_time_features(df, horizon)
        df = self._add_momentum_features(df, horizon)
        df = self._add_spike_features(df, horizon)
        df = self._add_volatility_features(df, horizon)
        df = self._add_horizon_features(df, horizon)
        
        # 添加目标变量
        df['target'] = self.df['price'].shift(-horizon)
        
        return df
    
    def build_all_horizons(self) -> dict:
        """
        为所有4个horizon构建特征集
        
        Returns
        -------
        dict
            {'h1': df1, 'h2': df2, 'h3': df3, 'h4': df4}
        """
        result = {}
        for name, horizon in self.HORIZONS.items():
            print(f"Building features for {name} (horizon={horizon})...")
            result[name] = self.build_features(horizon)
        return result
    
    def get_feature_names(self, horizon: int) -> dict:
        """
        获取特征名称分类（互斥分组，无重复计数）
        
        Returns
        -------
        dict
            按类别分组的特征名称
        """
        df = self.build_features(horizon)
        cols = set(df.columns.tolist())
        
        feature_groups = {}
        assigned = set()
        
        # 按优先级顺序分配（每个特征只归属一个类别）
        # 1. 原始价格和目标
        feature_groups['original'] = {'price'}
        assigned.update(feature_groups['original'])
        
        feature_groups['target'] = {'target'}
        assigned.update(feature_groups['target'])
        
        # 2. 滞后特征
        feature_groups['lag'] = {c for c in cols - assigned if 'lag' in c}
        assigned.update(feature_groups['lag'])
        
        # 3. 同时刻统计
        feature_groups['same_time'] = {c for c in cols - assigned if 'same_' in c}
        assigned.update(feature_groups['same_time'])
        
        # 4. 尖峰特征
        feature_groups['spike'] = {c for c in cols - assigned if any(x in c for x in ['spike', 'percentile', 'intervals_since'])}
        assigned.update(feature_groups['spike'])
        
        # 5. 波动率特征
        feature_groups['volatility'] = {c for c in cols - assigned if any(x in c for x in ['volatility', 'cv_', 'amplitude'])}
        assigned.update(feature_groups['volatility'])
        
        # 6. 动量特征
        feature_groups['momentum'] = {c for c in cols - assigned if any(x in c for x in ['diff_', 'pct_', 'momentum', 'acceleration', 'vs_'])}
        assigned.update(feature_groups['momentum'])
        
        # 7. 滚动统计（排除已分配的）
        feature_groups['rolling'] = {c for c in cols - assigned if any(x in c for x in ['mean_', 'std_', 'max_', 'min_', 'median_', 'skew_', 'range_'])}
        assigned.update(feature_groups['rolling'])
        
        # 8. Horizon特征
        feature_groups['horizon'] = {c for c in cols - assigned if any(x in c for x in ['horizon', 'target_', 'crosses_'])}
        assigned.update(feature_groups['horizon'])
        
        # 9. 时间特征（剩余的大部分是时间特征）
        feature_groups['time'] = cols - assigned
        
        # 转换为排序列表
        feature_groups = {k: sorted(list(v)) for k, v in feature_groups.items()}
        
        # 统计
        print("\n特征统计（互斥分组）:")
        print("-" * 40)
        total = 0
        for group, features in feature_groups.items():
            print(f"{group:12s}: {len(features):3d} 个特征")
            total += len(features)
        print("-" * 40)
        print(f"{'总计':12s}: {total:3d} 个特征")
        print(f"{'实际列数':10s}: {len(df.columns):3d} 列")
        
        return feature_groups


def load_data(filepath: str) -> pd.DataFrame:
    """
    加载原始数据
    
    Parameters
    ----------
    filepath : str
        CSV文件路径
    
    Returns
    -------
    pd.DataFrame
        原始数据
    """
    df = pd.read_csv(filepath)
    df.columns = ['date', 'hour', 'interval', 'price']
    return df


def prepare_train_test(
    df: pd.DataFrame,
    train_end: str = '2021-12-31 23:59:59',
    val_end: str = '2023-12-31 23:59:59'
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    划分训练集、验证集、测试集
    
    Parameters
    ----------
    df : pd.DataFrame
        特征DataFrame
    train_end : str
        训练集截止日期
    val_end : str
        验证集截止日期
    
    Returns
    -------
    Tuple
        (train_df, val_df, test_df)
    """
    train = df[df.index <= train_end]
    val = df[(df.index > train_end) & (df.index <= val_end)]
    test = df[df.index > val_end]
    
    print(f"训练集: {train.index.min()} ~ {train.index.max()}, {len(train):,} 条")
    print(f"验证集: {val.index.min()} ~ {val.index.max()}, {len(val):,} 条")
    print(f"测试集: {test.index.min()} ~ {test.index.max()}, {len(test):,} 条")
    
    return train, val, test

In [ ]:
if __name__ == "__main__":

    # 获取数据源
    df = load_data('./raw_data/processed_data.csv')
    
    print(f"数据形状: {df.shape}")
    print(f"数据范围: {df.date.values[0]} ~ {df.date.values[-1]}")
    print("\n原始数据示例:")
    print(pd.concat([df.head(2), df.tail(2)]))
    
    # 初始化特征工程器
    print("\n初始化特征工程器...")
    fe = ERCOTFeatureEngineer(df)

    
    print(f"\n特征DataFrame形状: {df_h1.shape}")
    print("\n特征列表:")
    fe.get_feature_names(horizon=1)
    
    # 所有4个horizon特征
    print("进行特征提取")
    for name, horizon in ERCOTFeatureEngineer.HORIZONS.items():
        df_fea = fe.build_features(horizon)
        df_fea = df_fea.dropna()
        df_fea.to_csv(f'train_data/train_data{horizon}.csv', index=None)

数据形状: (432388, 4)
数据范围: 01/01/2011 ~ 12/31/2024

原始数据示例:
              date  hour  interval  price
0       01/01/2011   1.0       1.0  25.11
1       01/01/2011   1.0       2.0  23.45
432386  12/31/2024  24.0       3.0  19.78
432387  12/31/2024  24.0       4.0  18.78

初始化特征工程器...

特征DataFrame形状: (432388, 119)

特征列表:

特征统计（互斥分组）:
----------------------------------------
original    :   1 个特征
target      :   1 个特征
lag         :  10 个特征
same_time   :   3 个特征
spike       :  18 个特征
volatility  :   7 个特征
momentum    :  14 个特征
rolling     :  36 个特征
horizon     :   6 个特征
time        :  22 个特征
----------------------------------------
总计          : 118 个特征
实际列数      : 118 列
进行特征提取
